In [ ]:
#test_proj tests projections and roi's
#Grok prompt: python geemap clip an image to an aoi in UTM8N coordinates
#Then: get  roi = Map.user_roi in UTM8N coordinates instead of WGS84
#Then: more coaching through a few errors.
#OBSOLETE NOTE (I pressed on). This may not be worth it.
#CONCLUSION: works for UTM8N

## OBSOLETE method uses GeoDataFrame

In [ ]:
#To get the Region of Interest (ROI) drawn on a geemap map in UTM Zone 8N (EPSG:32608) coordinates instead of WGS84 (EPSG:4326), 
#you need to reproject the Map.user_roi geometry from WGS84 to UTM8N. The geemap library and Google Earth Engine (GEE) handle 
#geometries in WGS84 by default, so you'll transform the coordinates after retrieving the ROI. Below is a Python script that 
#demonstrates how to achieve this.
import ee
import geemap
import geopandas as gpd
from shapely.geometry import shape

# Initialize the Earth Engine module
ee.Initialize()

In [ ]:
# Create an interactive map
Map = geemap.Map()

# Load an example image (e.g., Landsat 8) to display on the map
#image = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') #somewhere in Greenland
#         .filterDate('2023-01-01', '2023-12-31')
#         .first())
image = ee.Image("LANDSAT/LC08/C02/T1_L2/LC08_060019_20140606") #Johns Hopkins

# Visualization parameters
vis_params = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],  # Red, Green, Blue
    'min': 0,
    'max': 30000,
    'gamma': 1.4
}

# Add the image to the map
Map.addLayer(image, vis_params, 'Landsat 8')
Map.centerObject(image, 8)  # Center the map on the image
Map

In [ ]:
# Instructions: Draw a polygon on the map using the drawing tools, then run the next part

# Get the user-drawn ROI
roi = Map.user_roi
if roi is None:
    print("Please draw a polygon on the map and run again.")
else:
    # Convert the EE geometry to GeoJSON
    geo_json = roi.getInfo()
    print(geo_json)
    print(geo_json['type'])

    ## Ensure GeoJSON is in the correct format
    #if geo_json['type'] == 'Feature':
    #    geometry = shape(geo_json['geometry'])
    #else:
    #    raise ValueError("Unexpected GeoJSON format. Expected a Feature with geometry.")

    # Check if the GeoJSON type is Polygon
    if geo_json['type'] == 'Polygon':
        geometry = shape(geo_json)  # Convert GeoJSON Polygon to Shapely geometry
    else:
        raise ValueError(f"Unexpected GeoJSON type: {geo_json['type']}. Expected 'Polygon'.")

    # Create a GeoDataFrame with the geometry (in WGS84, EPSG:4326)
    gdf = gpd.GeoDataFrame(index=[0], geometry=[geometry], crs="EPSG:4326")

    # Reproject the GeoDataFrame to UTM Zone 8N (EPSG:32608)
    gdf_utm = gdf.to_crs("EPSG:32608")

In [ ]:
#gdf_utm
geometry
type(geometry)

In [ ]:
    # Extract the geometry in UTM8N coordinates
    utm_geometry = gdf_utm.geometry.iloc[0]
    type(utm_geometry)

In [ ]:
    utm_coords = list(utm_geometry.exterior.coords)  # Get the coordinates of the polygon
    utm_coords
    type(utm_coords) #list
    type(utm_coords[0][0])


In [ ]:
    print("ROI coordinates in UTM Zone 8N (EPSG:32608):")
    for coord in utm_coords:
        print(f"Easting: {coord[0]:.2f}, Northing: {coord[1]:.2f}")

In [ ]:
    # Optional: Clip an image to the ROI and display
    ee_geometry = geemap.geopandas_to_ee(gdf_utm)
    clipped_image = image.clip(ee_geometry)
    Map.addLayer(clipped_image, vis_params, 'Clipped Landsat 8')
    Map.centerObject(ee_geometry, 10)
    Map
    

## OBSOLETE another way - grok again

In [ ]:
import pandas as pd
import pyproj
from pyproj import Transformer

# Load the DataFrame
try:
    df = pd.read_csv('regions.csv')
except FileNotFoundError:
    print("Warning: 'regions.csv' not found. Creating default CSV.")
    data = {
        'RegionName': ['Amazon_Rainforest', 'Sahara_Desert', 'Himalayas', 'Great_Barrier_Reef', 'Yosemite'],
        'MinLon': [-70.0, 0.0, 80.0, 145.0, -119.8],
        'MaxLon': [-50.0, 25.0, 90.0, 155.0, -119.4],
        'MinLat': [-10.0, 15.0, 25.0, -25.0, 37.5],
        'MaxLat': [0.0, 30.0, 35.0, -15.0, 38.0]
    }
    df = pd.DataFrame(data)
    df.to_csv('regions.csv', index=False)

In [ ]:
# Define source and target coordinate systems
source_crs = 'EPSG:4326'  # WGS84 (lat/lon)
target_crs = 'EPSG:32608'  # UTM Zone 10N (example, suitable for Yosemite)

# Initialize transformer
transformer = Transformer.from_crs(source_crs, target_crs, always_xy=True)

# Function to transform coordinates
def transform_bbox(min_lon, max_lon, min_lat, max_lat):
    # Transform the four corners of the bounding box
    # Bottom-left (min_lon, min_lat)
    x1, y1 = transformer.transform(min_lon, min_lat)
    # Top-right (max_lon, max_lat)
    x2, y2 = transformer.transform(max_lon, max_lat)
    # Return min/max in the new coordinate system
    return min(x1, x2), max(x1, x2), min(y1, y2), max(y1, y2)

# Transform coordinates for each region
transformed_regions = []
for index, row in df.iterrows():
    min_x, max_x, min_y, max_y = transform_bbox(row['MinLon'], row['MaxLon'], row['MinLat'], row['MaxLat'])
    transformed_regions.append({
        'RegionName': row['RegionName'],
        'MinX': min_x,
        'MaxX': max_x,
        'MinY': min_y,
        'MaxY': max_y
    })

# Create a new DataFrame with transformed coordinates
transformed_df = pd.DataFrame(transformed_regions)

# Format coordinates to 0 decimal places 
transformed_df['MinX'] = transformed_df['MinX'].round(0)
transformed_df['MaxX'] = transformed_df['MaxX'].round(0)
transformed_df['MinY'] = transformed_df['MinY'].round(0)
transformed_df['MaxY'] = transformed_df['MaxY'].round(0)

# Save to a new CSV
transformed_df.to_csv('regions_transformed.csv', index=False)
print("\nTransformed regions saved to 'regions_transformed.csv':")
print(transformed_df)

# Example: Format first row's coordinates as a comma-separated string
first_row = transformed_df.iloc[0]
formatted = f"{first_row['MinX']:.3f},{first_row['MaxX']:.3f},{first_row['MinY']:.3f},{first_row['MaxY']:.3f}"
print(f"\nFirst row coordinates (formatted): {formatted}")

## OBSOLETE another way - simpler?

In [ ]:
import geemap
import ee
import pyproj
from shapely.geometry import box
from shapely.ops import transform

# Initialize Earth Engine
ee.Initialize()

# Define the input bounding box in WGS84 (lon, lat)
# Example: a bounding box in WGS84 (minx, miny, maxx, maxy)
wgs84_bbox = [-135.0, 57.0, -134.0, 58.0]

# Step 1: Transform WGS84 to UTM Zone 8
wgs84 = pyproj.CRS("EPSG:4326")  # WGS84
utm8 = pyproj.CRS("EPSG:32608")  # UTM Zone 8N
project = pyproj.Transformer.from_crs(wgs84, utm8, always_xy=True).transform

# Convert WGS84 bbox to a Shapely geometry
wgs84_box = box(wgs84_bbox[0], wgs84_bbox[1], wgs84_bbox[2], wgs84_bbox[3])

# Transform to UTM Zone 8
utm8_box = transform(project, wgs84_box)

# Step 2: Regularize the bounding box in UTM (ensure it's a rectangle)
# The transformed box may be skewed; we take the envelope to make it rectangular
utm8_rect = utm8_box.envelope
utm8_bounds = utm8_rect.bounds  # (minx, miny, maxx, maxy) in UTM

#TODO: would be awesome to register corners to Landsat pixel corners, but that's too much for now.

# Step 3: Transform back to WGS84
project_back = pyproj.Transformer.from_crs(utm8, wgs84, always_xy=True).transform
wgs84_rect = transform(project_back, utm8_rect)
wgs84_new_bounds = wgs84_rect.bounds  # (minx, miny, maxx, maxy) in WGS84

#NOTE: I actually want the wgs84_rect as the output, not wgs84_new_bounds

# Print results
print("Original WGS84 BBox:", wgs84_bbox)
print("UTM Zone 8 rect:", utm8_rect)
print("UTM Zone 8 BBox:", utm8_bounds)
print('WGS84 rect:',wgs84_rect)
print("Regularized WGS84 BBox:", wgs84_new_bounds)

In [ ]:
print(type(wgs84_rect))
print(type(wgs84_rect_ee))

In [ ]:
# Optional: Create an Earth Engine geometry for visualization
ee_geometry_old = ee.Geometry.Rectangle(wgs84_bbox)
ee_geometry = ee.Geometry.Rectangle(wgs84_new_bounds)
#shapely back to ee_polygon: Get the exterior coordinates as a list of lists
wgs84_rectcoords = [list(wgs84_rect.exterior.coords)]
# If the polygon has holes (interiors), add them
for interior in wgs84_rect.interiors:
    wgs84_rectcoords.append(list(interior.coords))
wgs84_rect_ee = ee.Geometry.Polygon(wgs84_rectcoords)

ee_map = geemap.Map()
ee_map.addLayer(ee_geometry_old, {}, "Old BBox")
ee_map.addLayer(ee_geometry, {}, "Regularized BBox")
ee_map.addLayer(wgs84_rect_ee, {}, "Reg. BBox WGS84_rect_ee")
ee_map.centerObject(ee_geometry, 8)
ee_map  # Display the map

#Original WGS84 BBox: [-135.0, 57.0, -134.0, 58.0]
#UTM Zone 8 BBox: (500000.0, 6317385.920695577, 560746.6236577407, 6429147.611133263)
#Regularized WGS84 BBox: (-135.0, 56.996006464924804, -133.97228067948123, 58.003929218471185)

## same as above but in function form - this is what worked!

In [ ]:
import geemap
import ee
import pyproj
import pandas as pd
from shapely.geometry import box
from shapely.ops import transform

def transform_regularize_bbox(wgs84_bbox):
    """
    Transform a WGS84 bounding box to UTM Zone 8N, regularize it to a rectangle,
    and transform it back to WGS84.
    
    Args:
        wgs84_bbox (list or tuple): [minx, miny, maxx, maxy] in WGS84 (lon, lat)
    
    Returns:
        tuple: Regularized bounding box [minx, miny, maxx, maxy] in WGS84
    """
    # Initialize Earth Engine (optional, only if visualization is needed)
    ee.Initialize()

    # Define CRS for WGS84 and UTM Zone 8N
    wgs84 = pyproj.CRS("EPSG:4326")
    utm8 = pyproj.CRS("EPSG:32608")
    
    # Create transformers
    project_to_utm = pyproj.Transformer.from_crs(wgs84, utm8, always_xy=True).transform
    project_to_wgs84 = pyproj.Transformer.from_crs(utm8, wgs84, always_xy=True).transform

    # Convert WGS84 bbox to Shapely geometry
    wgs84_box = box(wgs84_bbox[0], wgs84_bbox[1], wgs84_bbox[2], wgs84_bbox[3])

    # Transform to UTM Zone 8N
    utm8_box = transform(project_to_utm, wgs84_box)

    # Regularize to a rectangle in UTM
    utm8_rect = utm8_box.envelope

    # Transform back to WGS84
    wgs84_rect = transform(project_to_wgs84, utm8_rect)
    wgs84_new_bounds = wgs84_rect.bounds  # (minx, miny, maxx, maxy)

#    return wgs84_new_bounds #NOTE: I actually want the wgs84_rect as the output, not wgs84_new_bounds
    return wgs84_rect

# Example usage
wgs84_bbox = [-135.0, 57.0, -134.0, 58.0]
wgs84_bbox_new = transform_regularize_bbox(wgs84_bbox)

print("Original WGS84 BBox:", wgs84_bbox)
print("Regularized WGS84 BBox:", wgs84_bbox_new)
#Original WGS84 BBox: [-135.0, 57.0, -134.0, 58.0]
#OBSOLETE: Regularized WGS84 BBox (new_bounds): (-135.0, 56.996006464924804, -133.97228067948123, 58.003929218471185)
#Regularized WGS84 BBox: POLYGON ((-135 57.00000000000001, -134.0001071119298 56.996006464924804, -133.97228067948123 57.999779130695565, -135 58.003929218471185, -135 57.00000000000001))

In [ ]:
print(type(wgs84_bbox_new))
print(type(wgs84_bbox))

In [ ]:
coords = list(wgs84_bbox_new.exterior.coords)
# Create a DataFrame with x, y columns
df = pd.DataFrame(coords, columns=['x', 'y'])
# Save to CSV
#df.to_csv('test_proj.csv', index=False)

In [ ]:
# Load the CSV file
df = pd.read_csv(r'C:\Users\andyb\Documents\U\SEAN_Glacier-Dynamics\glacierPropsLandsat.csv')
#df.iloc[0]

In [ ]:
#OBSOLETE:
#transformed_regions = []
#for index, row in df.iterrows():
#    min_x, min_y, max_x, max_y = transform_regularize_bbox([row['LonMin'], row['LatMin'], row['LonMax'], row['LatMax']])
#    transformed_regions.append({
#        'Name': row['Name'],
#        'LonMinX': min_x,
#        'LonMaxX': max_x,
#        'LatMinY': min_y,
#        'LatMaxY': max_y
#    })
#transformed_regions

In [ ]:
#OBSOLETE test run for 1 row
row=df.iloc[1]
print(row)
poly = transform_regularize_bbox([row['LonMin'], row['LatMin'], row['LonMax'], row['LatMax']])

coords=list(poly.exterior.coords)
print(coords)

coord_dict = {}
for i, (x, y) in enumerate(coords, 1):
    coord_dict[f'x{i}'] = x
    coord_dict[f'y{i}'] = y
print(coord_dict)
print(type(coord_dict))

tr=[]
tr.append(coord_dict)
print(tr)
tr.append(coord_dict)
print(tr)

asdf=pd.DataFrame(tr)
print(asdf)

In [ ]:
transformed_regions = []
for index, row in df.iterrows():
    poly = transform_regularize_bbox([row['LonMin'], row['LatMin'], row['LonMax'], row['LatMax']])
    # Extract exterior coordinates
    coords=list(poly.exterior.coords)
    # Flatten coordinates into a single row with columns x1, y1, x2, y2, ...
    coord_dict = {}
    coord_dict['Name']=row['Name']
    for i, (x, y) in enumerate(coords, 1):
        coord_dict[f'x{i}'] = x
        coord_dict[f'y{i}'] = y
    transformed_regions.append(coord_dict)

print(transformed_regions[0:3])

In [ ]:
dft = pd.DataFrame(transformed_regions)

In [ ]:
#combined_df = pd.concat([df, dft], axis=1)
#print(combined_df) #Name is repeated
#41           La Perouse -137.360000 -137.160000  58.390000  58.590000
#41           La Perouse -137.373442 -137.147765  58.386474  58.593513
#df_joined=df.join(dft) #ValueError: columns overlap but no suffix specified
#print(df_joined)
df_merge=pd.merge(df,dft,on='Name')
print(df_merge[0:3])

In [ ]:
#write to csv
df_merge.to_csv(r'C:\Users\andyb\Documents\U\SEAN_Glacier-Dynamics\glacierPropsLandsat.csv',index=False)

## OBSOLETE another way - cartoee

In [ ]:
#From Google AI (has a few errors), prompt: set projection of geemap
#Method 2: The cartoee module, which works with geemap and cartopy, offers greater flexibility for creating static, publication-quality maps with custom projections. This method is ideal when you need to output a map in a specific projection for a journal or report. 
#Example: Using cartoee with an Equal Earth projection
import ee
import geemap
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from geemap import cartoee

# Initialize Earth Engine
#ee.Initialize()

In [ ]:
# Get a sample Earth Engine image (e.g., MODIS NDVI)
image = (
    ee.ImageCollection('MODIS/MCD43A4_006_NDVI')
    .filter(ee.Filter.date('2022-05-01', '2022-06-01'))
    .select("NDVI")
    .first()
)

# Define visualization parameters for the image
vis_params = {'min': 0.0, 'max': 1.0} #, 'palette': 'ndvi'} #KeyError: 'cannot provide `palette` in vis_params if `cmap` is specified' and then:
#Color is not a valid CSS 3.0 color ('FF0000' or 'red' for red). Found: 'ndvi'.".

# Define the region of interest and the desired projection
bbox = [180, -88, -180, 88] # Global bounds. Was -180 first which yielded reversed map.
projection = ccrs.EqualEarth()

In [ ]:
# Create a Matplotlib figure
fig = plt.figure(figsize=(15, 10))

# Use cartoee to generate a map with the custom projection
ax = geemap.cartoee.get_map(
    image,
    region=bbox,
    proj=projection,
    vis_params=vis_params,
    #cmap="ndvi", #KeyError: 'cannot provide `palette` in vis_params if `cmap` is specified'
)

# Add a color bar and title
cb = geemap.cartoee.add_colorbar(ax, vis_params=vis_params, loc='right')
ax.set_title(label='MODIS NDVI (Equal Earth Projection)', fontsize=15)

# Display the plot
plt.show()